In [1]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Crippen, Descriptors, Lipinski, rdMolDescriptors

In [ ]:
df = pd.read_csv("../datasets/B3DB_classification.csv").rename(columns={"compound_name": "Name"})
df["Class"] = df["BBB+/BBB-"].map({"BBB+": 1, "BBB-": -1})
df = df[["NO.", "Name", "SMILES", "logBB", "Class"]]


bad = df["NO."].isin([5044, 7738])
df.loc[bad, "SMILES"] = df.loc[bad, "SMILES"].str.replace("[C+](", "C(", regex=False)

mols = df["SMILES"].map(Chem.MolFromSmiles)

In [ ]:
df[["MW", "nHBAcc", "nHBDon", "SLogP", "TopoPSA"]] = [
    [Descriptors.ExactMolWt(m), Lipinski.NumHAcceptors(m), Lipinski.NumHDonors(m), Crippen.MolLogP(m), rdMolDescriptors.CalcTPSA(m)]
    for m in mols
]

df[["MW", "SLogP", "TopoPSA"]] = df[["MW", "SLogP", "TopoPSA"]].round(2)
df[["nHBAcc", "nHBDon"]] = df[["nHBAcc", "nHBDon"]].astype(int)

df.to_csv("../datasets/B3DB_classification_annotated.csv", index=False)
df.head()

,NO.,Name,SMILES,logBB,Class,MW,nHBAcc,nHBDon,SLogP,TopoPSA
0,1,sulphasalazine,O=C(O)c1cc(N=Nc2ccc(S(=O)(=O)Nc3ccccn3)cc2)ccc1O,-2.69,-1,398.07,7,3,3.70,141.31
1,2,moxalactam,COC1(NC(=O)C(C(=O)O)c2ccc(O)cc2)C(=O)N2C(C(=O)...,-2.52,-1,520.10,11,4,-1.13,206.30
2,3,clioquinol,Oc1c(I)cc(Cl)c2cccnc12,-2.40,-1,304.91,2,1,3.20,33.12
3,4,bbcpd11 (cimetidine analog) (y-g13),CCNC(=NCCSCc1ncccc1Br)NC#N,-2.15,-1,341.03,4,2,2.11,73.10
4,5,schembl614298,CN1CC[C@]23c4c5ccc(OC6O[C@H](C(=O)O)[C@@H](O)[...,-2.15,-1,461.17,9,5,-1.24,149.15


In [17]:
# Import the removed metals dataset – removed manually:
import pandas as pd
import numpy as np
import itertools
from rdkit import Chem
from rdkit.Chem import AllChem
import networkx as nx

import seaborn as sns
import matplotlib.pyplot as plt
from feature_selection import structural_similarity, mw_diff

full_dataset = df
smiles = full_dataset['SMILES']

# Check validity by generating fp using rdkit.
invalid = []
fpgen = AllChem.GetRDKitFPGenerator()
for smile_id, smile in enumerate(smiles): 
    try: 
        mol = Chem.MolFromSmiles(smile)
        fpgen.GetFingerprint(mol)
    except: 
        invalid.append(smile_id)

# Remove invalid structures. 
all_valid_dataset = full_dataset.drop(invalid, axis=0)
all_valid_dataset = all_valid_dataset.reset_index()
smiles_valid = all_valid_dataset['SMILES']
logBB_valid = all_valid_dataset['logBB']
bbb_class_valid = all_valid_dataset['Class']

print('Dataset size after removing metals and unstandardizable smiles structures: ', len(all_valid_dataset))

# Calculate structural similarity and MW to identify stereoisomers.
sim_vals, _, _ = structural_similarity(smiles_valid)
sim_matrix_valid = np.array(sim_vals).reshape((len(smiles_valid), len(smiles_valid)))
mw_diff_matrix = np.array(mw_diff(smiles_valid)).reshape((len(smiles_valid), len(smiles_valid)))

sim_inds, mw_inds= np.column_stack(np.where(sim_matrix_valid == 1)),  np.column_stack(np.where(mw_diff_matrix == 0))
sim_inds, mw_inds = np.array([np.array([x[0], x[1]]) for x in sim_inds if x[0]<x[1]]), np.array([np.array([x[0], x[1]]) for x in mw_inds if x[0]<x[1]])
drop_inds = [ list(el) for el in sim_inds if el in mw_inds]

# Find clusters of stereoisomers and select the one with the lowest index (most likely to have a logBB value)
G = nx.Graph()
G.add_edges_from(drop_inds)
clusters = list(nx.connected_components(G))
drop_ind_clusters = [sorted(list(c))[1:] for c in clusters]
drop_ind_clusters_flat = [item for sublist in drop_ind_clusters for item in sublist]

# Remove all redundant stereoisomers
all_valid_dataset_no_str = all_valid_dataset.drop(drop_ind_clusters_flat, axis=0)
all_valid_dataset_no_str = all_valid_dataset_no_str.reset_index()


Dataset size after removing metals and unstandardizable smiles structures:  7807


In [18]:
len(all_valid_dataset_no_str)

4016

In [ ]:
output_file = '../datasets/druglike_b3db.csv'

base = all_valid_dataset_no_str


drug_like_mask = (
    base["MW"].le(800)
    & base["nHBDon"].le(6)
    & base["nHBAcc"].le(11)
    & base["SLogP"].le(5)
    & base["SLogP"].gt(0)
    & base["TopoPSA"].lt(180)
)

all_valid_dataset_drug_like = base.loc[drug_like_mask].copy()
all_valid_dataset_not_drug_like = base.loc[~drug_like_mask].copy()

all_valid_dataset_drug_like.to_csv(output_file,index=False,)

print("Cleaned valid dataset:", len(base))
print("Drug-like subset:", len(all_valid_dataset_drug_like))
print("Exact complement:", len(all_valid_dataset_not_drug_like))

Cleaned valid dataset: 4016
Drug-like subset: 3156
Exact complement: 860
